In [1]:
import math
import torch
import torch.nn.functional as F

# =====================================================
# OFFICIAL ALIBI SLOPES
# =====================================================
def get_slopes_power_of_2(n):
    start = 2 ** (-2 ** -(math.log2(n) - 3))
    ratio = start
    return [start * (ratio ** i) for i in range(n)]

def get_alibi_slopes(n_heads):
    """ Official ALiBi slope generation handling non-power-of-2 heads """
    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))

    closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
    slopes_1 = get_slopes_power_of_2(closest_power_of_2)
    slopes_2 = get_slopes_power_of_2(2 * closest_power_of_2)
    slopes_2 = slopes_2[0::2][: n_heads - closest_power_of_2]
    return torch.tensor(slopes_1 + slopes_2)

# =====================================================
# BUILD ALIBI BIAS MATRIX (CAUSAL & BIDIRECTIONAL)
# =====================================================
def build_alibi_bias(n_heads, seq_len, causal=True, device="cpu"):
    """
    Returns an ALiBi bias tensor of shape [1, n_heads, seq_len, seq_len].
    
    Causal:        bias[i, j] = -m * (i - j)  for j <= i
    Bidirectional: bias[i, j] = -m * abs(i - j)
    """
    slopes = get_alibi_slopes(n_heads).to(device)
    positions = torch.arange(seq_len, device=device)
    
    # 1. Compute physical token distance matrix
    # rows = query index (i), cols = key index (j)
    distance = positions[:, None] - positions[None, :]
    
    if causal:
        # For causal, we care about distance to past tokens (i >= j).
        # We don't clamp to 0 here; we let the downstream causal mask 
        # handle future tokens completely.
        alibi_bias = -slopes[:, None, None] * distance[None, :, :]
    else:
        # For bidirectional, distance is symmetric absolute physical offset
        abs_distance = torch.abs(distance)
        alibi_bias = -slopes[:, None, None] * abs_distance[None, :, :]

    return alibi_bias.unsqueeze(0)

# =====================================================
# EXAMPLE RUN & VERIFICATION
# =====================================================
if __name__ == "__main__":
    B, T, n_heads, head_dim = 1, 4, 2, 8  # Reduced sizes for readable printout
    
    print("Slopes Generated ---")
    slopes = get_alibi_slopes(n_heads)
    for h, m in enumerate(slopes):
        print(f"Head {h}: slope = {m:.4f}")
    print()

    # Generate biases
    causal_bias = build_alibi_bias(n_heads, seq_len=T, causal=True)
    bidir_bias  = build_alibi_bias(n_heads, seq_len=T, causal=False)

    # Toy Attention pass
    Q = torch.randn(B, n_heads, T, head_dim)
    K = torch.randn(B, n_heads, T, head_dim)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(head_dim)

    # Inject Causal ALiBi + Apply standard causal mask
    causal_scores = scores + causal_bias
    mask = torch.tril(torch.ones(T, T))
    causal_scores = causal_scores.masked_fill(mask == 0, float("-inf"))
    print(causal_scores[0, 0])  # Print scores for head 0 after bias + mask
    
    attention = F.softmax(causal_scores, dim=-1)
    print("Final Causal Attention Matrix Shape ---")
    print(attention.shape)

Slopes Generated ---
Head 0: slope = 0.0625
Head 1: slope = 0.0039

tensor([[ 0.5590,    -inf,    -inf,    -inf],
        [-1.0101,  0.6438,    -inf,    -inf],
        [-0.8259,  2.3822,  1.2218,    -inf],
        [-1.4661,  0.7103,  0.0860,  0.1687]])
Final Causal Attention Matrix Shape ---
torch.Size([1, 2, 4, 4])
